In [2]:
import vrep 
import sys
import time 
import numpy as np
from tank import *
import skfuzzy 
from skfuzzy import control as ctrl

In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

def find_place(dist_EN, dist_ES):
    en = ctrl.Antecedent(np.arange(0, 6.01, 0.01), 'EN')
    es = ctrl.Antecedent(np.arange(0, 6.01, 0.01), 'ES')
    speed = ctrl.Consequent(np.arange(-1, 8.01, 0.01), 'speed')

    en['close'] = fuzz.trapmf(en.universe, [0, 0, 1.5, 1.55])
    en['mid'] = fuzz.trapmf(en.universe, [1.5, 1.55, 2.5, 2.51])
    en['far'] = fuzz.trapmf(en.universe, [2.5, 2.51, 6, 6])

    es['close'] = fuzz.trapmf(es.universe, [0, 0, 1.1, 2])
    es['far'] = fuzz.trapmf(es.universe, [1.1, 2, 6, 6])

    speed['stop'] = fuzz.trimf(speed.universe, [-1, 0, 1])
    speed['break'] = fuzz.trimf(speed.universe, [0, 3, 5])
    speed['go'] = fuzz.trimf(speed.universe, [5, 6, 7])

    rules = [
        ctrl.Rule(en['far'] & es['far'], speed['go']),
        ctrl.Rule(en['close'] & es['close'], speed['go']),
        ctrl.Rule(en['close'] & es['far'], speed['go']),
        ctrl.Rule(en['far'] & es['close'], speed['break']),
        ctrl.Rule(en['mid'] & es['far'], speed['break']),
        ctrl.Rule(en['mid'] & es['close'], speed['stop']),
    ]

    sim_ctrl = ctrl.ControlSystem(rules)
    sim = ctrl.ControlSystemSimulation(sim_ctrl)

    sim.input['EN'] = dist_EN
    sim.input['ES'] = dist_ES
    sim.compute()

    return sim.output['speed']


def set_to_park(dist_ES, dist_SE):
    es = ctrl.Antecedent(np.arange(0, 6.01, 0.01), 'ES')
    se = ctrl.Antecedent(np.arange(0, 6.01, 0.01), 'SE')
    speed = ctrl.Consequent(np.arange(-1, 8.01, 0.01), 'speed')

    se['close'] = fuzz.trapmf(se.universe, [0, 0, 2.4, 3])
    se['far'] = fuzz.trapmf(se.universe, [2.4, 3, 6, 6])

    es['close'] = fuzz.trapmf(es.universe, [0, 0, 1.1, 2])
    es['far'] = fuzz.trapmf(es.universe, [1.1, 2, 6, 6])

    speed['stop'] = fuzz.trimf(speed.universe, [-1, 0, 1])
    speed['break'] = fuzz.trimf(speed.universe, [0, 3, 5])
    speed['go'] = fuzz.trimf(speed.universe, [5, 6, 7])

    rules = [
        ctrl.Rule(se['far'] & es['far'], speed['go']),
        ctrl.Rule(se['far'] & es['close'], speed['stop']),
        ctrl.Rule(se['close'] & es['far'], speed['break']),
        ctrl.Rule(se['close'] & es['close'], speed['go'])
    ]

    sim_ctrl = ctrl.ControlSystem(rules)
    sim = ctrl.ControlSystemSimulation(sim_ctrl)

    sim.input['SE'] = dist_SE
    sim.input['ES'] = dist_ES
    sim.compute()

    return sim.output['speed']


def drive_in_place(dist_ES, dist_SE):
    min_dist = min(dist_ES, dist_SE)

    dist = ctrl.Antecedent(np.arange(0, 6.01, 0.01), 'dist')
    speed = ctrl.Consequent(np.arange(-1, 8.01, 0.01), 'speed')

    dist['close'] = fuzz.trapmf(dist.universe, [0, 0, 0.79, 0.80])
    dist['far'] = fuzz.trapmf(dist.universe, [0.79, 0.80, 6, 6])

    speed['stop'] = fuzz.trimf(speed.universe, [-1, 0, 1])
    speed['break'] = fuzz.trimf(speed.universe, [0, 1, 2])
    speed['go'] = fuzz.trimf(speed.universe, [5, 6, 7])

    rules = [
        ctrl.Rule(dist['far'], speed['go']),
        ctrl.Rule(dist['close'], speed['stop']),
    ]

    sim_ctrl = ctrl.ControlSystem(rules)
    sim = ctrl.ControlSystemSimulation(sim_ctrl)

    sim.input['dist'] = min_dist
    sim.compute()

    return sim.output['speed']


def parallel_park(dist_SW, dist_WS):
    min_dist = dist_SW

    dist = ctrl.Antecedent(np.arange(0, 6.01, 0.01), 'dist')
    speed = ctrl.Consequent(np.arange(-1, 7.01, 0.01), 'speed')

    dist['close'] = fuzz.trapmf(dist.universe, [0, 0, 1.1, 2])
    dist['far'] = fuzz.trapmf(dist.universe, [1.1, 2, 6, 6])

    speed['stop'] = fuzz.trimf(speed.universe, [-1, 0, 1])
    speed['break'] = fuzz.trimf(speed.universe, [0, 1, 2])
    speed['go'] = fuzz.trimf(speed.universe, [5, 6, 7])

    rules = [
        ctrl.Rule(dist['far'], speed['go']),
        ctrl.Rule(dist['close'], speed['stop']),
    ]

    sim_ctrl = ctrl.ControlSystem(rules)
    sim = ctrl.ControlSystemSimulation(sim_ctrl)

    sim.input['dist'] = min_dist
    sim.compute()

    return sim.output['speed']


In [7]:
vrep.simxFinish(-1) # closes all opened connections, in case any prevoius wasnt finished
clientID=vrep.simxStart('127.0.0.1',19999,True,True,5000,5) # start a connection

if clientID!=-1:
    print ("Connected to remote API server")
else:
    print("Not connected to remote API server")
    sys.exit("Could not connect")

#create instance of Tank
tank=Tank(clientID)

Connected to remote API server


In [8]:
proximity_sensors=["EN","ES","NE","NW","SE","SW","WN","WS"]
proximity_sensors_handles=[0]*8

# get handle to proximity sensors
for i in range(len(proximity_sensors)):
    err_code,proximity_sensors_handles[i] = vrep.simxGetObjectHandle(clientID,"Proximity_sensor_"+proximity_sensors[i], vrep.simx_opmode_blocking)
    
#read and print values from proximity sensors
#first reading should be done with simx_opmode_streaming, further with simx_opmode_buffer parameter
for sensor_name, sensor_handle in zip(proximity_sensors,proximity_sensors_handles):
        err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector=vrep.simxReadProximitySensor(clientID,sensor_handle,vrep.simx_opmode_streaming)

In [ ]:
tank.forward(5)

#continue reading and printing values from proximity sensors
distances = dict()
detection_states = dict()

for sensor_name, sensor_handle in zip(proximity_sensors,proximity_sensors_handles):
        err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector=vrep.simxReadProximitySensor(clientID,sensor_handle,vrep.simx_opmode_buffer)
        distances[sensor_name] = np.linalg.norm(detectedPoint)
        detection_states[sensor_name] = detectionState

stages = [
     'find_place',
     'set_to_park',
     'drive_in_place',
     'parallel_park',
     'finish_park'
]

t = time.time()
stage = 'find_place'
while (time.time()-t)<100: 
    for sensor_name, sensor_handle in zip(proximity_sensors,proximity_sensors_handles):
        err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector=vrep.simxReadProximitySensor(clientID,sensor_handle,vrep.simx_opmode_buffer )
        if(err_code == 0):
            # print("Proximity_sensor_"+sensor_name, np.linalg.norm(detectedPoint))
            distances[sensor_name] = np.linalg.norm(detectedPoint)
            detection_states[sensor_name] = detectionState
            for dist, state in zip(distances, detection_states):
                # print(f"{dist}, {detection_states[dist]}")
                pass
                 
            # if(detection_states['ES'] or detection_states['SE']):
            #     print(distances['SE'], distances['ES'])
            #     print(detection_states['SE'], detection_states['ES'])

    if stage == 'find_place':
        tank_speed = find_place(distances['EN'], distances['ES'])
        # print(tank_speed)
        if tank_speed < 2:
            tank.forward(0)
            stage = 'set_to_park'
        else:
            tank.forward(tank_speed)
    if stage == 'set_to_park':
        tank_speed = set_to_park(distances['ES'], distances['SE'])
        # print(tank_speed)
        if tank_speed < 1:
            tank.forward(0)
            stage = 'drive_in_place'
            print("Zmieniam na faze parkowania")
        else:
            tank.forward(tank_speed)
    if stage == 'drive_in_place':
        tank_speed = drive_in_place(distances['ES'],distances['SE'])
        # print(tank_speed)
        if tank_speed < 0.3:
            tank.forward(0)
            stage = 'parallel_park'
            print("Zmieniam faze na parallel")
        else:
            tank.leftvelocity = -tank_speed
            tank.rightvelocity = -tank_speed / 6
            tank.setVelocity()
    if stage == 'parallel_park':
        tank_speed = parallel_park(distances['SW'], distances['WS'])
        # print(tank_speed)
        if tank_speed < 0.18:
            # tank.stop()
            stage = 'finish_park'
            print("Zmieniam na finalną fazę")
        else:
            tank.leftvelocity = -tank_speed / (2.333)
            tank.rightvelocity = -tank_speed
            tank.setVelocity()
    if stage == 'finish_park':
        # tank.forward(1)
        tank.leftvelocity = 1
        tank.rightvelocity = 0.8
        tank.setVelocity()
        if not detection_states['EN'] and not detection_states['ES'] and distances['NE'] < 0.8:
            tank.stop()
            break
        
    # print()

Zmieniam na faze parkowania
Zmieniam faze na parallel
Zmieniam na finalną fazę
